# Discourse Diversity Analysis: Shannon Entropy & Jensen-Shannon Divergence

This notebook implements the text-based discourse diversity analysis described in:
> **"Curated Voice: Communicative Inequality and Platform Power in NFT and DeFi Communities on X"**

### Overview
We compare the **Central-Actor-Associated Discourse** (highly connected, high in-degree nodes) against **Regular Actor Discourse** (the broader community) using two information-theoretic measures:
- **Shannon Entropy** (Shannon, 1948) — quantifies lexical diversity within each group
- **Jensen-Shannon Divergence** (Lin, 1991) — measures how different the two groups' discourse distributions are

### Preprocessing Pipeline
Raw tweet text undergoes multi-stage preprocessing:
- URLs, @handles, RT markers, numbers, and punctuation are removed
- Hashtag symbol `#` is removed
- Text is lowercased

### Actor Classification
Central-Actor are identified by weighted degree in the retweet/mention network (top 3). For community-specific analyses, central actors may alternatively be defined as specific named accounts with the highest structural centrality (e.g., as reported in the paper's network analysis section).

**References:**  
Shannon, C. E. (1948). A mathematical theory of communication. *Bell System Technical Journal*, 27(3), 379–423.  
Lin, J. (1991). Divergence measures based on the Shannon entropy. *IEEE Transactions on Information Theory*, 37(1), 145–151.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
from scipy.spatial.distance import jensenshannon
from collections import Counter
import re
import warnings
import networkx as nx
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported!")
print(f"📌 Pandas: {pd.__version__}")
print(f"📌 NetworkX: {nx.__version__}")

In [ ]:
from google.colab import files

print("="*60)
print("📤 STEP 1: Upload TWEET DATA")
print("="*60)
uploaded_tweets = files.upload()
tweet_filename = list(uploaded_tweets.keys())[0]
print(f"✅ Tweet file: {tweet_filename}")

print("\n" + "="*60)
print("📤 STEP 2: Upload EDGELIST")
print("="*60)
uploaded_edges = files.upload()
edge_filename = list(uploaded_edges.keys())[0]
print(f"✅ Edgelist file: {edge_filename}")

In [ ]:
print("\n" + "="*60)
print("📊 LOADING DATA")
print("="*60)

# Load tweets
df_tweets = pd.read_csv(tweet_filename)
print(f"✅ Tweets loaded: {df_tweets.shape[0]:,} tweets")
print(f"   Unique users: {df_tweets['username'].nunique():,}")

# Load edgelist
df_edges = pd.read_csv(edge_filename)
print(f"\n✅ Edgelist loaded: {len(df_edges):,} interactions")
print(f"   Columns: {df_edges.columns.tolist()}")

In [ ]:
print("\n" + "="*60)
print("⚙️ CONFIGURE EDGELIST COLUMNS")
print("="*60)
print("Current columns:", df_edges.columns.tolist())
print("\n👇 Adjust these variables to match YOUR column names:")

# Change these to YOUR actual column names!
SOURCE_COL = 'source'    # Column with "who does the action" (retweeter, mentioner)
TARGET_COL = 'target'    # Column with "who receives action" (gets retweeted/mentioned)
WEIGHT_COL = 'weight'    # Column with interaction count (or None if unweighted)

print(f"\n✅ Using:")
print(f"   Source: {SOURCE_COL}")
print(f"   Target: {TARGET_COL}")
print(f"   Weight: {WEIGHT_COL}")

# Verify columns exist
if SOURCE_COL not in df_edges.columns:
    print(f"❌ ERROR: Column '{SOURCE_COL}' not found!")
    print(f"   Available: {df_edges.columns.tolist()}")
if TARGET_COL not in df_edges.columns:
    print(f"❌ ERROR: Column '{TARGET_COL}' not found!")

In [ ]:
print("\n" + "="*60)
print("🕸️ SOCIAL NETWORK ANALYSIS (TOP 3 BREAK-POINT THRESHOLD)")
print("="*60)

# Build directed network
print("Building network...")
G = nx.DiGraph()

if WEIGHT_COL and WEIGHT_COL in df_edges.columns:
    for _, row in df_edges.iterrows():
        source = row[SOURCE_COL]
        target = row[TARGET_COL]
        weight = float(row[WEIGHT_COL]) if pd.notna(row[WEIGHT_COL]) else 1.0

        if G.has_edge(source, target):
            G[source][target]['weight'] += weight
        else:
            G.add_edge(source, target, weight=weight)

    total_degree = dict(G.degree(weight='weight'))
else:
    for _, row in df_edges.iterrows():
        source = row[SOURCE_COL]
        target = row[TARGET_COL]
        G.add_edge(source, target)

    total_degree = dict(G.degree())

print(f"✅ Network built!")
print(f"   Nodes: {G.number_of_nodes():,}")
print(f"   Edges: {G.number_of_edges():,}")

# Sort all actors by total weighted degree in descending order
degree_sorted = sorted(total_degree.items(), key=lambda x: x[1], reverse=True)

# ---------------------------------------------------------------
# FIXED ACTOR CLASSIFICATION: TOP 3 BREAK-POINT THRESHOLD
# As specified in the manuscript's supplementary material,
# the top 3 accounts represent the core structural elite.
# ---------------------------------------------------------------
# Get the usernames of the top 3 central actors
central_actors_list = [user for user, deg in degree_sorted[:3]]
# Everyone else is a regular actor
regular_actors_list = [user for user, deg in degree_sorted[3:]]

# Establish the cut-off value at Rank 3 for reporting
threshold_value = degree_sorted[2][1] if len(degree_sorted) >= 3 else degree_sorted[-1][1]

print(f"\n✅ Actor classification complete (Threshold: Top 3 Empirical Break-Point)")
print(f"   CENTRAL ACTORS:  {len(central_actors_list):,} users (Rank 1 to 3)")
print(f"   REGULAR ACTORS:  {len(regular_actors_list):,} users (Rank 4+)")
print(f"   Break-point degree value (Rank 3): {threshold_value:.2f}")

# Show top 10 central actors to verify the steep drop between Rank 3 and Rank 4
print(f"\n🏆 Top 10 actors by total weighted degree:")
for i, (user, deg) in enumerate(degree_sorted[:10], 1):
    marker = "⭐️ [CENTRAL CORE]" if i <= 3 else "  [REGULAR ACTOR]"
    print(f"   {i:2d}. @{user:<25s} {deg:>10,.1f} {marker}")

In [ ]:
print("\n" + "="*60)
print("🔍 FILTERING TWEETS BY ACTOR TYPE (SANITIZED MATCHING)")
print("="*60)

# 1. Sanitize tweet usernames (lowercase, strip whitespace, remove '@')
df_tweets['username_clean'] = df_tweets['username'].astype(str).str.lower().str.strip().str.replace('@', '', regex=False)

# 2. Sanitize actor lists from the network analysis step
central_actors_clean = set([str(user).lower().strip().replace('@', '') for user in central_actors_list])
regular_actors_clean = set([str(user).lower().strip().replace('@', '') for user in regular_actors_list])

# 3. Filter using the cleaned columns
df_central = df_tweets[df_tweets['username_clean'].isin(central_actors_clean)].copy()
df_regular = df_tweets[df_tweets['username_clean'].isin(regular_actors_clean)].copy()

print(f"Central actors' tweets: {len(df_central):,}")
print(f"Regular actors' tweets: {len(df_regular):,}")

# 4. Calculate coverage
total_matched = len(df_central) + len(df_regular)
coverage = (total_matched / len(df_tweets)) * 100 if len(df_tweets) > 0 else 0
print(f"\nCoverage: {coverage:.1f}%")

if coverage < 80:
    print(f"⚠️ WARNING: Low coverage ({coverage:.1f}%) - check if one file uses IDs and the other uses Handles!")

    # Quick debug helper printout
    print("\n💡 [DEBUG INFO] Inspecting format differences:")
    print(f"   Sample from df_tweets['username']: {df_tweets['username'].dropna().iloc[:3].tolist() if not df_tweets['username'].empty else 'Empty'}")
    print(f"   Sample from central_actors_list:  {list(central_actors_list)[:3] if central_actors_list else 'Empty'}")


In [ ]:
print("\n" + "="*60)
print("🧹 PREPROCESSING TEXT")
print("="*60)

def preprocess_text(text, remove_stopwords=True):
    """
    Multi-stage text preprocessing pipeline.
    - Removes URLs, @handles, RT markers, numbers, punctuation
    - Retains hashtag text (removes # symbol only)
    - Lowercases text
    - Removes English stop words and domain-level background terms
      (crypto, blockchain, web3, defi, nft, opensea, ethereum, eth, btc, bitcoin)
      to prevent common background terminology from dominating frequency distributions
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)           # Remove # but keep hashtag text
    text = re.sub(r'\brt\b', '', text)      # Remove retweet markers
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        # Domain-level background terms excluded to prevent common terminology
        # from dominating frequency distributions and obscuring thematic variation
        domain_stop = {
            'crypto', 'blockchain', 'web3', 'defi', 'nft',
            'opensea', 'ethereum', 'eth', 'btc', 'bitcoin'
        }
        stop_words.update(domain_stop)
        words = text.split()
        text = ' '.join([w for w in words if w not in stop_words])

    return text

print("Processing central actors' tweets...")
df_central['clean_text'] = df_central['full_text'].apply(preprocess_text)

print("Processing regular actors' tweets...")
df_regular['clean_text'] = df_regular['full_text'].apply(preprocess_text)

print("✅ Preprocessing complete!")

In [ ]:
print("\n" + "="*60)
print("📈 GINI COEFFICIENT (Network Inequality - Total Weighted Degree)")
print("="*60)

def calculate_gini(values):
    """Calculate Gini coefficient to measure inequality of total weighted degree distribution."""
    sorted_values = np.sort(values)
    n = len(values)
    if n == 0 or np.sum(sorted_values) == 0:
        return 0
    cumsum = np.cumsum(sorted_values)
    gini = (2 * np.sum([(i+1) * val for i, val in enumerate(sorted_values)])) / (n * cumsum[-1]) - (n+1)/n
    return gini

# CHANGE: Swapped out 'in_degrees' for 'total_degree.values()' calculated in the previous step
total_degrees = list(total_degree.values())
gini = calculate_gini(total_degrees)

print(f"Gini Coefficient: {gini:.4f}")

if gini > 0.7:
    print("🔴 EXTREME inequality in network activity distribution")
elif gini > 0.4:
    print("🟡 MODERATE inequality in network activity distribution")
else:
    print("🟢 LOW inequality in network activity distribution")

# Top 5% concentration
sorted_degrees = sorted(total_degrees, reverse=True)
top_5_count = int(len(sorted_degrees) * 0.05)

# Safety check: ensure at least 1 user is selected if the community size is small
if top_5_count == 0 and len(sorted_degrees) > 0:
    top_5_count = 1

top_5_sum = sum(sorted_degrees[:top_5_count])
total_sum = sum(sorted_degrees)
top_5_pct = (top_5_sum / total_sum) * 100 if total_sum > 0 else 0

print(f"Top 5% of actors control: {top_5_pct:.1f}% of all network activity (interactions)")


In [ ]:
print("\n" + "="*60)
print("📚 SHANNON ENTROPY")
print("="*60)
print("Measures lexical diversity: higher entropy = more diverse, multifaceted discourse")
print("Lower entropy = homogeneous, topically concentrated discourse\n")

def calculate_entropy(text_series):
    """
    Calculate Shannon entropy of unigram term frequency distribution.
    H(X) = -sum(p(x) * log2(p(x)))
    Returns: (entropy, vocabulary_size, total_word_count)
    """
    corpus = ' '.join(text_series.dropna().astype(str))
    words = corpus.split()

    if not words:
        return 0, 0, 0

    word_counts = Counter(words)
    total_words = len(words)
    probabilities = [count / total_words for count in word_counts.values()]
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)

    return entropy, len(word_counts), total_words

H_central, vocab_ca, words_ca = calculate_entropy(df_central['clean_text'])
H_regular, vocab_ra, words_ra = calculate_entropy(df_regular['clean_text'])

print(f"CENTRAL ACTORS (central-actor-associated discourse):")
print(f"  Shannon Entropy H:  {H_central:.4f} bits")
print(f"  Vocabulary size:    {vocab_ca:,} unique terms")
print(f"  Total word count:   {words_ca:,}")

print(f"\nREGULAR ACTORS (broader community discourse):")
print(f"  Shannon Entropy H:  {H_regular:.4f} bits")
print(f"  Vocabulary size:    {vocab_ra:,} unique terms")
print(f"  Total word count:   {words_ra:,}")

diff = H_regular - H_central
print(f"\nEntropy difference (Regular − Central): {diff:+.4f} bits")

In [ ]:
print("\n" + "="*60)
print("🎯 JENSEN-SHANNON DIVERGENCE")
print("="*60)
print("Measures discourse distance between central-actor and regular-actor distributions")
print("JSD range: 0.0 (identical) → 1.0 (completely divergent)\n")

def get_distribution(text_series, vocab):
    """
    Compute unigram term frequency distribution over a shared vocabulary.
    Returns probability distribution (normalized counts).
    """
    corpus = ' '.join(text_series.dropna().astype(str))
    words = corpus.split()
    word_counts = Counter(words)
    counts = np.array([word_counts[w] for w in vocab])
    total = counts.sum()
    return counts / total if total > 0 else np.ones(len(vocab)) / len(vocab)

# Build shared vocabulary from top-1000 terms in each group
# (union of both groups to capture full discourse space)
corpus_ca = ' '.join(df_central['clean_text'].dropna().astype(str))
corpus_ra = ' '.join(df_regular['clean_text'].dropna().astype(str))

counter_ca = Counter(corpus_ca.split())
counter_ra = Counter(corpus_ra.split())

vocab_ca_top = set([w for w, _ in counter_ca.most_common(1000)])
vocab_ra_top = set([w for w, _ in counter_ra.most_common(1000)])
shared_vocab = sorted(vocab_ca_top | vocab_ra_top)

print(f"Shared vocabulary size: {len(shared_vocab):,} terms")

dist_central = get_distribution(df_central['clean_text'], shared_vocab)
dist_regular = get_distribution(df_regular['clean_text'], shared_vocab)

jsd = jensenshannon(dist_central, dist_regular)

print(f"\nJSD (Central-Actor-Associated Discourse vs. Regular Actor Discourse): {jsd:.4f}")

if jsd < 0.2:
    print("🚨 VERY SIMILAR — central-actor discourse closely mirrors community discourse (curated voice pattern)")
elif jsd < 0.4:
    print("🟡 MODERATELY SIMILAR — partial overlap in discourse patterns")
else:
    print("✅ DISTINCT — central-actor discourse differs substantially from regular community discourse")


In [ ]:
print("\n" + "="*60)
print("📊 CREATING VISUALIZATIONS")
print("="*60)

fig = plt.figure(figsize=(18, 6))

# Plot 1: Lorenz Curve (network inequality)
ax1 = plt.subplot(1, 3, 1)
sorted_norm = np.sort(in_degrees) / sum(in_degrees)
cumsum = np.cumsum(sorted_norm)
x = np.linspace(0, 1, len(cumsum))

ax1.plot(x, cumsum, 'b-', linewidth=2, label='Actual')
ax1.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Equality')
ax1.fill_between(x, cumsum, x, alpha=0.3, color='pink')
ax1.set_title(f'Lorenz Curve (Gini = {gini:.3f})', fontsize=14, fontweight='bold')
ax1.set_xlabel('Cumulative % of Users', fontsize=12)
ax1.set_ylabel('Cumulative % of Engagement', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Plot 2: Shannon Entropy Comparison
ax2 = plt.subplot(1, 3, 2)
bars = ax2.bar(
    ['Central Actors', 'Regular Actors'],
    [H_central, H_regular],
    color=['coral', 'skyblue'],
    edgecolor='black',
    linewidth=2
)
ax2.set_title('Lexical Diversity\n(Shannon Entropy)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Shannon Entropy (bits)', fontsize=12)

for bar, val in zip(bars, [H_central, H_regular]):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.grid(axis='y', alpha=0.3)

# Plot 3: JSD Gauge
ax3 = plt.subplot(1, 3, 3, projection='polar')
theta = np.linspace(0, np.pi, 100)

colors = ['lightcoral', 'lightyellow', 'lightgreen']
bounds = [0, 0.3, 0.7, 1.0]

for i in range(len(bounds)-1):
    mask = (theta >= bounds[i]*np.pi) & (theta <= bounds[i+1]*np.pi)
    ax3.fill_between(theta[mask], 0, 1, color=colors[i], alpha=0.5)

jsd_angle = jsd * np.pi
ax3.arrow(0, 0, jsd_angle, 0.8, width=0.03, head_width=0.1,
         head_length=0.1, fc='red', ec='red', linewidth=3)
ax3.plot([jsd_angle], [0.85], 'ro', markersize=15)

ax3.set_ylim(0, 1)
ax3.set_xticks([0, np.pi/2, np.pi])
ax3.set_xticklabels(['0.0\nAligned', '0.5\nModerate', '1.0\nDivergent'], fontsize=10)
ax3.set_yticks([])
ax3.set_title(f'JSD = {jsd:.3f}\n(Central vs. Regular Actors)', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("✅ Visualization complete!")


In [ ]:
print("\n" + "="*60)
print("📋 COMPREHENSIVE RESULTS REPORT")
print("="*60)

print(f"\n🎯 DISCOURSE PATTERN DETECTED:")
print("="*60)

# Pattern classification based on entropy + JSD combination
curated   = (H_regular > 4.0 and jsd < 0.3)
echo      = (H_regular < 3.0 and jsd < 0.3)
genuine   = (H_regular > 4.0 and jsd > 0.7)

if curated:
    pattern = "🚨 CURATED VOICE"
    explanation = "High entropy + Low JSD: regular actors show apparent diversity but mirror central actor framing"
elif echo:
    pattern = "🔴 ECHO CHAMBER"
    explanation = "Low diversity + High alignment: concentrated, homogeneous discourse throughout community"
elif genuine:
    pattern = "✅ GENUINE DIVERSITY"
    explanation = "High diversity + Independent discourse: central and regular actors discuss distinct topics"
else:
    pattern = "🟡 MIXED PATTERN"
    explanation = "Results indicate partial alignment; further qualitative analysis recommended"

print(f"{pattern}")
print(f"{explanation}\n")

print(f"📊 FULL METRICS SUMMARY:")
print("="*60)
print(f"Gini Coefficient:                       {gini:.4f}")
print(f"Top 5% actor concentration:             {top_5_pct:.1f}% of amplification")
print(f"Shannon Entropy — Central Actors:       {H_central:.4f} bits")
print(f"Shannon Entropy — Regular Actors:       {H_regular:.4f} bits")
print(f"Entropy Difference (Regular − Central): {diff:+.4f} bits")
print(f"Jensen-Shannon Divergence (JSD):        {jsd:.4f}")
print("="*60)
